# First-order ODEs: analytical solution and finite-difference method

-- by GPT 5.2

We will study a simple but very important first-order ODE:

$$
\frac{dc}{dt} = -k(c-c_{eq}),
$$

which describes **relaxation toward equilibrium**.

This type of equation appears in many materials contexts:

- concentration relaxation,
- thermal equilibration,
- defect annealing,
- simple kinetic models.

We will:

1. Solve the ODE analytically with SymPy
2. Interpret the solution physically
3. Solve it numerically with a finite-difference method
4. Compare exact and numerical solutions
5. Study stability and time-step effects

## 1. Imports and setup

In [ ]:
import sympy as sp
sp.init_printing()

import numpy as np
import matplotlib.pyplot as plt

## 2. Define the ODE symbolically

We consider

$$
\frac{dc}{dt} = -k(c-c_{eq}),
$$

where

- $c(t)$ is the evolving quantity,
- $c_{eq}$ is the equilibrium value,
- $k>0$ is the relaxation rate.


In [ ]:
t = sp.symbols('t', real=True)
k, c_eq = sp.symbols('k c_eq', positive=True, real=True)
c = sp.Function('c')

ode = sp.Eq(sp.diff(c(t), t), -k*(c(t) - c_eq))
ode

## 3. Solve the ODE analytically with SymPy

In [ ]:
sol = sp.dsolve(ode)
sol

The general solution should have the form

$$
c(t) = c_{eq} + C e^{-kt}.
$$

To determine the constant, impose the initial condition

$$
c(0)=c_0.
$$

In [ ]:
c0 = sp.symbols('c0', real=True)

C1 = sp.symbols('C1')
general_sol = sol.rhs

eq_for_C1 = sp.Eq(general_sol.subs(t, 0), c0)
C1_sol = sp.solve(eq_for_C1, C1)[0]
C1_sol

In [ ]:
c_exact = sp.simplify(general_sol.subs(C1, C1_sol))
c_exact

So the exact analytical solution is

$$
c(t)=c_{eq} + (c_0-c_{eq})e^{-kt}.
$$

This is one of the most important solutions in kinetics and transport.

## 4. Verify the analytical solution

We now check that the exact solution satisfies the ODE.

In [ ]:
check_exact = sp.simplify(sp.diff(c_exact, t) + k*(c_exact - c_eq))
check_exact

If the result is `0`, the analytical solution is verified.

## 5. Physical interpretation

The solution

$$
c(t)=c_{eq} + (c_0-c_{eq})e^{-kt}
$$

means:

- the deviation from equilibrium decays exponentially,
- larger $k$ means faster relaxation.

At long times,

$$
\lim_{t\to\infty} c(t)=c_{eq}.
$$

In [ ]:
long_time_limit = sp.limit(c_exact, t, sp.oo)
long_time_limit

## 6. Plot the exact solution for several values of \(k\)

In [ ]:
t_vals = np.linspace(0, 5, 400)
c0_val = 1.0
c_eq_val = 0.2
k_vals = [0.5, 1.0, 2.0]

for kval in k_vals:
    cvals = c_eq_val + (c0_val - c_eq_val) * np.exp(-kval * t_vals)
    plt.plot(t_vals, cvals, label=f"k={kval}")

plt.xlabel("t")
plt.ylabel("c(t)")
plt.title("Exact analytical solution of first-order relaxation ODE")
plt.legend()
plt.show()

## 7. Finite-difference method (forward Euler)

We now solve the same ODE numerically.

Starting from

$$
\frac{dc}{dt} = -k(c-c_{eq}),
$$

we approximate the derivative by a forward difference:

$$
\frac{c^{n+1}-c^n}{\Delta t} \approx -k(c^n-c_{eq}).
$$

So the update rule becomes

$$
c^{n+1} = c^n - k\Delta t (c^n-c_{eq}).
$$

This is the **forward Euler method**.

In [ ]:
def forward_euler_relaxation(k, c_eq, c0, dt, t_end):
    Nt = int(np.round(t_end / dt)) + 1
    t = np.linspace(0, dt*(Nt-1), Nt)
    c = np.zeros(Nt)
    c[0] = c0

    for n in range(Nt - 1):
        c[n+1] = c[n] - k * dt * (c[n] - c_eq)

    return t, c

## 8. Compare exact and finite-difference solutions

In [ ]:
k_val = 1.0
c_eq_val = 0.2
c0_val = 1.0
dt = 0.2
t_end = 5.0

t_num, c_num = forward_euler_relaxation(k_val, c_eq_val, c0_val, dt, t_end)

c_exact_num = c_eq_val + (c0_val - c_eq_val) * np.exp(-k_val * t_num)

plt.plot(t_num, c_exact_num, '-', lw=2, label="exact")
plt.plot(t_num, c_num, 'o--', label="finite difference")
plt.xlabel("t")
plt.ylabel("c(t)")
plt.title("Exact vs forward Euler finite-difference solution")
plt.legend()
plt.show()

## 9. Error analysis

We can compute the pointwise error

$$
\text{error}(t_n)=|c_{\text{num}}^n-c_{\text{exact}}(t_n)|.
$$

In [ ]:
error = np.abs(c_num - c_exact_num)

plt.plot(t_num, error, 'o-')
plt.xlabel("t")
plt.ylabel("absolute error")
plt.title("Error of forward Euler method")
plt.show()

print("Maximum error =", error.max())

## 10. Time-step effect

Let us compare several time steps.

A smaller $\Delta t$ should generally improve accuracy.

In [ ]:
k_val = 1.0
c_eq_val = 0.2
c0_val = 1.0
t_end = 5.0
dt_list = [0.5, 0.2, 0.05]

for dt in dt_list:
    t_num, c_num = forward_euler_relaxation(k_val, c_eq_val, c0_val, dt, t_end)
    plt.plot(t_num, c_num, 'o--', label=f"FD dt={dt}")

t_fine = np.linspace(0, t_end, 400)
c_fine = c_eq_val + (c0_val - c_eq_val) * np.exp(-k_val * t_fine)
plt.plot(t_fine, c_fine, 'k-', lw=2, label="exact")

plt.xlabel("t")
plt.ylabel("c(t)")
plt.title("Effect of time step on forward Euler accuracy")
plt.legend()
plt.show()

## 11. Stability discussion

For the error $e^n = c^n - c_{eq}$, the numerical scheme becomes

$$
e^{n+1} = (1-k\Delta t)e^n.
$$

So the amplification factor is

$$
g = 1-k\Delta t.
$$

For stability, we need

$$
|1-k\Delta t| < 1.
$$

This gives

$$
0 < k\Delta t < 2.
$$

If $k\Delta t$ is too large, the numerical solution becomes oscillatory or unstable.

In [ ]:
def plot_stability_examples():
    k_val = 1.0
    c_eq_val = 0.2
    c0_val = 1.0
    t_end = 8.0

    for dt in [0.5, 1.5, 2.5]:
        t_num, c_num = forward_euler_relaxation(k_val, c_eq_val, c0_val, dt, t_end)
        plt.plot(t_num, c_num, 'o--', label=f"dt={dt}, k*dt={k_val*dt}")

    plt.axhline(c_eq_val, linestyle='--', label="equilibrium")
    plt.xlabel("t")
    plt.ylabel("c(t)")
    plt.title("Forward Euler stability behavior")
    plt.legend()
    plt.show()

plot_stability_examples()

Interpretation:

- if $k\Delta t < 1$, the solution approaches equilibrium smoothly,
- if $1 < k\Delta t < 2$, the solution oscillates but still decays,
- if $k\Delta t > 2$, the method becomes unstable.

This is a very important idea for numerical methods in materials modeling.

## 12. Closed-form numerical update

Because the forward Euler update is so simple, the numerical solution itself can be written as

$$
c^n = c_{eq} + (c^0-c_{eq})(1-k\Delta t)^n,
$$

form 

$$
e^n = e^0 \big( 1- k\Delta t )^n.
$$

This is the discrete analogue of the exact exponential solution.

In [ ]:
n = sp.symbols('n', integer=True, nonnegative=True)
dt = sp.symbols('dt', positive=True)

c_discrete = c_eq + (c0 - c_eq) * (1 - k*dt)**n
c_discrete

## 13. Compare discrete and continuous decay factors

Exact solution:

$$
e^{-kt}
$$

Finite-difference solution:

$$
(1-k\Delta t)^n, \qquad t_n=n\Delta t.
$$

For small $\Delta t$, these are close because

$$
1-k\Delta t \approx e^{-k\Delta t}.
$$

In [ ]:
k_val = 1.0
dt_val = 0.2
n_vals = np.arange(0, 21)
t_vals = n_vals * dt_val

exact_factor = np.exp(-k_val * t_vals)
disc_factor = (1 - k_val * dt_val)**n_vals

plt.plot(t_vals, exact_factor, 'k-', lw=2, label='exact decay factor')
plt.plot(t_vals, disc_factor, 'o--', label='discrete decay factor')
plt.xlabel("t")
plt.ylabel("decay factor")
plt.title("Continuous vs discrete decay")
plt.legend()
plt.show()

## 14. Newton's cooling

A common physical example is Newton's cooling law:

$$
\frac{dT}{dt} = -h(T-T_\infty).
$$

This is mathematically identical to the relaxation equation studied here.

You can reinterpret:

- $c \to T$,
- $c_{eq}$ to $T_\infty$,
- $k \to h$.

So the same exact and numerical methods apply immediately.

## 15. Summary

In this notebook, we:

- solved a first-order ODE analytically using SymPy,
- verified the exact solution,
- interpreted exponential relaxation physically,
- solved the same ODE numerically using a finite-difference time-stepping method,
- compared exact and numerical solutions,
- studied accuracy and stability.

This is foundational material for later topics such as diffusion, heat conduction, and reaction kinetics.

## 16. Optional exercises

1. Change the initial condition and equilibrium value.
2. Study how the solution changes when \(k\) becomes very large or very small.
3. Verify the stability condition numerically for several values of \(k\Delta t\).
4. Rewrite the notebook using temperature instead of concentration.
5. Implement the backward Euler method and compare it with forward Euler.